In [1]:
import sys

assert sys.version_info >= (3, 10)

In [2]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [5]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [7]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "6_poissonequn_with_periodicbc"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)



In [8]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [9]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Define exact solution for validation
    Since the sine function will be same for -1 and 1  in the opp direction we will be using sine function for this periodic bc, that satisfies the basic thing for using a periodic bc

In [10]:
def exact_solution(x):
    return np.sin(np.pi * x)

In [14]:
geom = dde.geometry.Interval(-1, 1)


Define the differential equation and boundary conditions

In [15]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx + torch.pi**2 * torch.sin(torch.pi * x)


Now define the left (Dirichlet) and right (Periodic) boundary conditions

In [20]:
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def boundary_periodic(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 1.0)

bc_periodic_value = dde.icbc.PeriodicBC(
    geom, 0, boundary_periodic, derivative_order=0
)
bc_periodic_derivative = dde.icbc.PeriodicBC(
    geom, 0, boundary_periodic, derivative_order=1
)
bcs = [bc_periodic_value, bc_periodic_derivative]

creating the training data

In [21]:
observe_x = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise  = 0.05 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe  = dde.icbc.PointSetBC(observe_x, observe_y, component=0)

combining all the data

In [22]:
data = dde.data.PDE(
    geom, pde, [bc_left] + bcs + [observe],
    num_domain=3000, num_boundary=200,
    num_test= 500, anchors=observe_x
)

In [25]:
net = dde.nn.FNN([1, 256, 128, 64, 1], "tanh", "Glorot uniform")
model = dde.Model(data, net)

In [27]:
loss_weights = [1.0, 40.0, 40.0, 40.0, 500.0]

In [28]:
print("\nStage 1: Adam Optimizer")
model.compile("adam", lr=0.001, loss_weights=loss_weights,
              decay=("inverse time", 1000, 0.1))
losshistory, train_state = model.train(iterations=4000, display_every=500)

dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights = loss_weights)
losshistory, train_state = model.train(display_every=50)


Stage 1: Adam Optimizer
Compiling model...
'compile' took 1.775507 s

Training model...

Step      Train loss                                            Test loss                                             Test metric
0         [4.57e+01, 6.66e-01, 2.67e+00, 0.00e+00, 2.93e+02]    [4.89e+01, 6.66e-01, 2.67e+00, 0.00e+00, 2.93e+02]    []  
500       [2.23e-01, 1.13e-02, 1.42e-02, 8.91e-08, 8.75e-01]    [1.11e-01, 1.13e-02, 1.42e-02, 8.91e-08, 8.75e-01]    []  
1000      [1.35e-01, 9.10e-03, 1.05e-02, 2.27e-05, 8.85e-01]    [7.56e-02, 9.10e-03, 1.05e-02, 2.27e-05, 8.85e-01]    []  
1500      [6.37e-02, 1.14e-04, 1.54e-03, 1.82e-02, 1.05e+00]    [3.47e-02, 1.14e-04, 1.54e-03, 1.82e-02, 1.05e+00]    []  
2000      [3.65e-02, 8.59e-03, 9.53e-03, 7.79e-05, 8.78e-01]    [1.90e-02, 8.59e-03, 9.53e-03, 7.79e-05, 8.78e-01]    []  
2500      [2.77e-02, 8.45e-03, 8.01e-03, 1.61e-05, 8.71e-01]    [1.85e-02, 8.45e-03, 8.01e-03, 1.61e-05, 8.71e-01]    []  
3000      [2.59e-02, 8.46e-03, 7.84e-03, 1